# Step 01_01_02 — Schema Discovery: aoestats

**Phase:** 01 — Data Exploration
**Pipeline Section:** 01_01 — Data Acquisition & Source Inventory
**Dataset:** aoestats
**Question:** What columns exist in each file type, what are their data types, is the schema consistent across the temporal range, and do matches and players share structurally overlapping columns?
**Invariants applied:** #6 (reproducibility), #7 (no magic numbers), #9 (pipeline discipline)
**ROADMAP reference:** `src/rts_predict/games/aoe2/datasets/aoestats/reports/ROADMAP.md` Step 01_01_02

Full census: `discover_parquet_schemas()` on all matches and players Parquet files.
`discover_json_schema()` on `overview.json`. Cross-column-name comparison.
No DuckDB type proposals.

## Cell 1 — Imports, config, paths

In [1]:
import json
import logging
from pathlib import Path

from rts_predict.common.json_utils import discover_json_schema
from rts_predict.common.parquet_utils import discover_parquet_schemas
from rts_predict.common.notebook_utils import get_reports_dir
from rts_predict.games.aoe2.config import (
    AOESTATS_RAW_DIR,
    AOESTATS_RAW_MATCHES_DIR,
    AOESTATS_RAW_PLAYERS_DIR,
    AOESTATS_RAW_OVERVIEW_DIR,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

ARTIFACTS_DIR: Path = (
    get_reports_dir("aoe2", "aoestats")
    / "artifacts"
    / "01_exploration"
    / "01_acquisition"
)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PRIOR_ARTIFACT: Path = ARTIFACTS_DIR / "01_01_01_file_inventory.json"

logger.info("ARTIFACTS_DIR: %s", ARTIFACTS_DIR)

INFO:__main__:ARTIFACTS_DIR: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/01_acquisition


## Cell 2 — Load 01_01_01 artifact to get file lists

In [2]:
with PRIOR_ARTIFACT.open() as fh:
    inventory = json.load(fh)

subdir_info = {sd["name"]: sd for sd in inventory["subdirs"]}
logger.info("Subdirectories from inventory: %s", list(subdir_info.keys()))

INFO:__main__:Subdirectories from inventory: ['matches', 'overview', 'players']


## Cell 3 — File selection (full census for all file types)

In [3]:
matches_files = sorted(
    f for f in AOESTATS_RAW_MATCHES_DIR.iterdir()
    if f.suffix == ".parquet"
)
players_files = sorted(
    f for f in AOESTATS_RAW_PLAYERS_DIR.iterdir()
    if f.suffix == ".parquet"
)
overview_file = AOESTATS_RAW_OVERVIEW_DIR / "overview.json"

logger.info("matches Parquet files: %d", len(matches_files))
logger.info("players Parquet files: %d", len(players_files))
logger.info("overview.json exists: %s", overview_file.exists())

INFO:__main__:matches Parquet files: 172


INFO:__main__:players Parquet files: 171


INFO:__main__:overview.json exists: True


## Cell 4 — Schema discovery

Parquet: `discover_parquet_schemas()` on all matches and players files.
JSON: `discover_json_schema()` on `overview.json`.

In [4]:
logger.info("Running discover_parquet_schemas() on %d matches files...", len(matches_files))
matches_result = discover_parquet_schemas(matches_files)
logger.info(
    "matches: all_same=%s, files_checked=%d, variant_cols=%s",
    matches_result["all_files_same_schema"],
    matches_result["files_checked"],
    matches_result["variant_columns"],
)

INFO:__main__:Running discover_parquet_schemas() on 172 matches files...


INFO:__main__:matches: all_same=False, files_checked=172, variant_cols=['raw_match_type', 'started_timestamp']


In [5]:
logger.info("Running discover_parquet_schemas() on %d players files...", len(players_files))
players_result = discover_parquet_schemas(players_files)
logger.info(
    "players: all_same=%s, files_checked=%d, variant_cols=%s",
    players_result["all_files_same_schema"],
    players_result["files_checked"],
    players_result["variant_columns"],
)

INFO:__main__:Running discover_parquet_schemas() on 171 players files...


INFO:__main__:players: all_same=False, files_checked=171, variant_cols=['castle_age_uptime', 'feudal_age_uptime', 'imperial_age_uptime', 'opening', 'profile_id']


In [6]:
logger.info("Running discover_json_schema() on overview.json...")
overview_profiles = discover_json_schema([overview_file], max_sample_values=3) if overview_file.exists() else []
logger.info("overview.json root keys: %d", len(overview_profiles))

INFO:__main__:Running discover_json_schema() on overview.json...


INFO:__main__:overview.json root keys: 8


## Cell 5 — Schema consistency check + cross-column name comparison

In [7]:
matches_representative = matches_result["schemas"][0] if matches_result["schemas"] else {}
players_representative = players_result["schemas"][0] if players_result["schemas"] else {}

matches_col_names = {c["name"] for c in matches_representative.get("columns", [])}
players_col_names = {c["name"] for c in players_representative.get("columns", [])}

shared_col_names = sorted(matches_col_names & players_col_names)
matches_only_names = sorted(matches_col_names - players_col_names)
players_only_names = sorted(players_col_names - matches_col_names)

logger.info("matches columns: %d", len(matches_col_names))
logger.info("players columns: %d", len(players_col_names))
logger.info("shared column names: %d", len(shared_col_names))
logger.info("matches-only column names: %d", len(matches_only_names))
logger.info("players-only column names: %d", len(players_only_names))
logger.info("shared names: %s", shared_col_names)

INFO:__main__:matches columns: 17


INFO:__main__:players columns: 13


INFO:__main__:shared column names: 1


INFO:__main__:matches-only column names: 16


INFO:__main__:players-only column names: 12


INFO:__main__:shared names: ['game_id']


## Cell 6 — Write JSON artifact

In [8]:
overview_columns_out = [
    {
        "name": kp.key,
        "physical_type": sorted(kp.observed_types)[0] if len(kp.observed_types) == 1 else sorted(kp.observed_types),
        "nullable": kp.nullable,
        "frequency": kp.frequency,
        "total_samples": kp.total_samples,
        "sample_values": kp.sample_values,
    }
    for kp in overview_profiles
]

artifact = {
    "step": "01_01_02",
    "dataset": "aoestats",
    "sampling": {
        "strategy": "census",
        "total_files_in_dataset": inventory["total_files"],
        "files_checked": len(matches_files) + len(players_files) + (1 if overview_file.exists() else 0),
        "method": (
            "Full census for all Parquet files (metadata-only read via pyarrow.parquet.read_schema). "
            "Full census for overview.json (1 file, discover_json_schema)."
        ),
    },
    "file_types": [
        {
            "type": "parquet",
            "subdirectory": "matches",
            "files_in_subdirectory": len(matches_files),
            "files_checked": matches_result["files_checked"],
            "schema": {
                "columns": [
                    {
                        "name": c["name"],
                        "physical_type": c["arrow_type"],
                        "nullable": c["nullable"],
                    }
                    for c in matches_representative.get("columns", [])
                ],
                "total_columns": matches_representative.get("total_columns", 0),
            },
            "consistency": {
                "all_files_same_schema": matches_result["all_files_same_schema"],
                "variant_columns": matches_result["variant_columns"],
            },
        },
        {
            "type": "parquet",
            "subdirectory": "players",
            "files_in_subdirectory": len(players_files),
            "files_checked": players_result["files_checked"],
            "schema": {
                "columns": [
                    {
                        "name": c["name"],
                        "physical_type": c["arrow_type"],
                        "nullable": c["nullable"],
                    }
                    for c in players_representative.get("columns", [])
                ],
                "total_columns": players_representative.get("total_columns", 0),
            },
            "consistency": {
                "all_files_same_schema": players_result["all_files_same_schema"],
                "variant_columns": players_result["variant_columns"],
            },
        },
        {
            "type": "json",
            "subdirectory": "overview",
            "files_in_subdirectory": 1,
            "files_checked": 1 if overview_file.exists() else 0,
            "schema": {
                "columns": overview_columns_out,
                "total_columns": len(overview_columns_out),
                "nesting_depth": 1,
            },
            "consistency": {
                "all_files_same_schema": True,
                "variant_columns": [],
            },
        },
    ],
    "cross_comparison": {
        "matches_vs_players_column_name_overlap": {
            "shared_column_names": shared_col_names,
            "shared_count": len(shared_col_names),
            "matches_only_count": len(matches_only_names),
            "players_only_count": len(players_only_names),
            "matches_only_column_names": matches_only_names,
            "players_only_column_names": players_only_names,
            "method": "raw string comparison of column names only — no semantic interpretation",
        }
    },
}

out_json = ARTIFACTS_DIR / "01_01_02_schema_discovery.json"
with out_json.open("w") as fh:
    json.dump(artifact, fh, indent=2)
logger.info("Wrote JSON artifact: %s", out_json)

INFO:__main__:Wrote JSON artifact: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/01_acquisition/01_01_02_schema_discovery.json


## Cell 7 — Write Markdown artifact

In [9]:
md_lines = [
    "# Step 01_01_02 — Schema Discovery: aoestats",
    "",
    "**Phase:** 01 — Data Exploration  ",
    "**Pipeline Section:** 01_01 — Data Acquisition & Source Inventory  ",
    "**Dataset:** aoestats  ",
    "**Invariants applied:** #6, #7, #9  ",
    "",
    "## Sampling methodology",
    "",
    "Full census for all file types.",
    "",
    "| File type | Subdirectory | Files in dataset | Files checked | Method |",
    "|-----------|-------------|-----------------|--------------|--------|",
    f"| Parquet | `matches/` | {len(matches_files)} | {matches_result['files_checked']} | metadata-only (pyarrow.parquet.read_schema) |",
    f"| Parquet | `players/` | {len(players_files)} | {players_result['files_checked']} | metadata-only (pyarrow.parquet.read_schema) |",
    f"| JSON | `overview/` | 1 | {1 if overview_file.exists() else 0} | discover_json_schema (root keys) |",
    "",
    "## matches/ schema (Parquet)",
    "",
    f"**Total columns:** {matches_representative.get('total_columns', 0)}  ",
    f"**Schema consistency:** {matches_result['all_files_same_schema']}  ",
    "",
    "| Column | Arrow type | Nullable |",
    "|--------|-----------|----------|",
]

for c in matches_representative.get("columns", []):
    md_lines.append(f"| `{c['name']}` | {c['arrow_type']} | {c['nullable']} |")

md_lines += [
    "",
    "## players/ schema (Parquet)",
    "",
    f"**Total columns:** {players_representative.get('total_columns', 0)}  ",
    f"**Schema consistency:** {players_result['all_files_same_schema']}  ",
    "",
    "| Column | Arrow type | Nullable |",
    "|--------|-----------|----------|",
]

for c in players_representative.get("columns", []):
    md_lines.append(f"| `{c['name']}` | {c['arrow_type']} | {c['nullable']} |")

md_lines += [
    "",
    "## overview/ schema (JSON)",
    "",
    f"**Total root keys:** {len(overview_profiles)}  ",
    "",
    "| Key | Observed types | Nullable | Frequency / Total |",
    "|-----|----------------|----------|-------------------|",
]

for kp in overview_profiles:
    type_str = ", ".join(sorted(kp.observed_types))
    md_lines.append(f"| `{kp.key}` | {type_str} | {kp.nullable} | {kp.frequency} / {kp.total_samples} |")

md_lines += [
    "",
    "## Cross-comparison: matches vs players column names",
    "",
    f"Comparison is raw string matching of column names only — no semantic interpretation.",
    "",
    f"| Metric | Count |",
    f"|--------|-------|",
    f"| Shared column names | {len(shared_col_names)} |",
    f"| matches-only column names | {len(matches_only_names)} |",
    f"| players-only column names | {len(players_only_names)} |",
    "",
    "**Shared column names:**",
    "",
]
for name in shared_col_names:
    md_lines.append(f"- `{name}`")

md_lines += [
    "",
    "**matches-only column names:**",
    "",
]
for name in matches_only_names:
    md_lines.append(f"- `{name}`")

md_lines += [
    "",
    "**players-only column names:**",
    "",
]
for name in players_only_names:
    md_lines.append(f"- `{name}`")

md_lines += [
    "",
    "## Notes",
    "",
    "- No DuckDB type proposals in this step (deferred to ingestion design).",
    "- Step scope: `content` (file headers/schemas).",
]

out_md = ARTIFACTS_DIR / "01_01_02_schema_discovery.md"
out_md.write_text("\n".join(md_lines) + "\n", encoding="utf-8")
logger.info("Wrote Markdown artifact: %s", out_md)

INFO:__main__:Wrote Markdown artifact: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoestats/reports/artifacts/01_exploration/01_acquisition/01_01_02_schema_discovery.md
